# Statistical Inference: Wild Bootstrap, Equivalence and Romano–Wolf
## Additional inference for the two main designs

Three procedures supplement the arrival-window level analysis (notebook 06) and within-day double difference (notebook 05):

1. Wild cluster bootstrap with Rademacher weights and the null imposed, reducing reliance on asymptotic reference distributions.
2. TOST equivalence tests, reversing the question to ask which effect sizes can be EXCLUDED, plus a one-sided 95% upper confidence limit.
3. Romano–Wolf stepdown max-|t| adjustment for testing five pollutants simultaneously.

**Data access.** The SIMA measurements used here are not
redistributed in this repository (SIMA provided them for this study on the condition
that they are not disseminated); the repository holds the code, the official school
calendar and all derived results. This notebook reads files that notebooks 00 and 01
build from those measurements, and its configuration cell stops with a clear message
if they are missing. To rebuild them, request the annual workbooks from SIMA, place
them in `data_raw/` and run notebook 00 (its header gives the exact file format).

**Role in the analysis.** Primary inference: the wild cluster bootstrap, the Romano–Wolf correction and the equivalence tests (TOST, UCL95) reported in the article for the primary analysis. The analytical hierarchy of the whole repository is stated in the README.

## 1. Initial setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import json
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

contaminantes = ['CO', 'NO2', 'O3', 'PM10', 'PM2.5']

# Reference equivalence margins of 10% and 20%:
# these are chosen magnitude benchmarks,
# not estimated effects of specific policies.
deltas_referencia = [('ref_10pct', 10.0), ('ref_20pct', 20.0)]

B_BOOTSTRAP = 999
SEMILLA = 2026

output_dir = Path('../data_processed')
resultados_dir = output_dir / 'resultados_metodos'
graficos_dir = output_dir / 'graficos_metodos'

print('Libraries imported successfully')

# Check required inputs (restricted data; see heading)
# Input-data check (restricted data; see heading)
archivos_requeridos = [output_dir / 'gaps_franjas_hibrido.csv',
                       output_dir / 'gaps_franjas_MIMA.csv',
                       output_dir / 'panel_pico_hibrido.csv',
                       output_dir / 'panel_pico_MIMA.csv']
faltantes = [str(a) for a in archivos_requeridos if not a.exists()]
if faltantes:
    raise FileNotFoundError(
        ''
        ''
        "Files derived from the SIMA measurements are missing; they are not redistributed in this repository. "
        "Request the annual workbooks from SIMA, place them in data_raw/ and run notebooks 00 and 01. "
        f"Missing: {faltantes}")
print(f"Input data found: {len(archivos_requeridos)} files")

Libraries imported successfully
Input data found: 4 files


## 2. Functions: design matrix, bootstrap and adjustments

### Null-imposed wild cluster bootstrap

1. Fit the model UNDER $H_0$, imposing $\beta=0$, and retain restricted residuals $\tilde u$ and fitted values $X\tilde\theta$.
2. In replication $b$, draw ONE Rademacher sign PER WEEK, $w_g^{(b)}\in\{-1,+1\}$ with equal probability, and construct $y_{st}^{(b)}=X\tilde\theta+w_{g(t)}^{(b)}\tilde u_{st}$. Every observation in a week shares its sign, preserving within-cluster dependence.
3. Refit and retain the statistic. Report the exceedance count with the finite-replication correction $(r+1)/(B+1)$.

This reduces reliance on asymptotic clustered-sandwich approximations, especially with few or unbalanced clusters (Cameron, Gelbach and Miller, 2008). It does not remove the assumption of limited dependence between clusters.

### TOST: reversing the question

Instead of asking whether an effect differs from zero, ask whether effects as large as a reference percentage margin can be EXCLUDED. For a proportional margin $d$ (0.10 or 0.20), use the coefficient boundaries $L=\log(1-d)$ and $U=\log(1+d)$:

$$ H_{01}:\beta\le L,\qquad H_{02}:\beta\ge U. $$

Equivalence within the percentage interval is established only if BOTH one-sided nulls are rejected. The reported p-value is the larger of the two. These bounds are asymmetric in log units because the target margins are symmetric in percentages.

The one-sided 95% upper confidence limit is $\hat\beta+1.645SE$, transformed to percentages. It is a signed upper limit on widening, not an absolute-effect bound, and lies below the upper endpoint of a two-sided 95% interval. The article calls it UCL95. TOST p-values are not multiplicity-adjusted.

### Romano–Wolf adjustment for five pollutants

Apply stepdown testing to the bootstrap maximum absolute t statistic. Compare the largest observed $|t|$ with the bootstrap distribution of $\max_j|t_j^{(b)}|$, remove that hypothesis and repeat over the remaining set, enforcing monotonic adjusted p-values. Under the procedure's assumptions this controls the probability of any false rejection in the family. Unlike Bonferroni, it uses dependence among pollutant statistics.

In [2]:
def matriz_diseno(df):
    '\n    Preferred design: intercept, clase, zone, month and weekday.\n    '
    piezas = [pd.get_dummies(df['estacion'], prefix='est', drop_first=True).astype(float),
              pd.get_dummies(df['mes'], prefix='mes', drop_first=True).astype(float),
              pd.get_dummies(df['dow'], prefix='dow', drop_first=True).astype(float)]
    return pd.concat([pd.Series(1.0, index=df.index, name='const'),
                      df['clase'].astype(float)] + piezas, axis=1)

def preparar(df, columnas_y, columna_cluster):
    '\n    Precompute bootstrap projections, restricted residuals\n    (without clase, imposing the null) and cluster structure.\n    '
    X = matriz_diseno(df)
    Xv = X.values.astype(float)
    Y = df[columnas_y].values.astype(float)
    idx_clase = X.columns.get_loc('clase')
    grupos = pd.factorize(df[columna_cluster])[0]
    orden = np.argsort(grupos, kind='stable')
    Xv, Y, grupos = Xv[orden], Y[orden], grupos[orden]
    inicios = np.flatnonzero(np.r_[True, grupos[1:] != grupos[:-1]])
    G = len(inicios)
    n, k = Xv.shape
    A = np.linalg.inv(Xv.T @ Xv)
    P = A @ Xv.T
    w_clase = Xv @ A[:, idx_clase]
    conservar = [j for j in range(k) if j != idx_clase]
    Xr = Xv[:, conservar]
    Br = np.linalg.lstsq(Xr, Y, rcond=None)[0]
    ajuste_restringido = Xr @ Br
    residuos_restringidos = Y - ajuste_restringido
    correccion = (G / (G - 1)) * ((n - 1) / (n - k))
    return dict(X=Xv, Y=Y, idx_clase=idx_clase, inicios=inicios, G=G, n=n, k=k,
                P=P, w_clase=w_clase, ajuste_restringido=ajuste_restringido,
                residuos_restringidos=residuos_restringidos, correccion=correccion)

def estadisticos(Y, d):
    "\n    'clase' coefficients, clustered variance, equation-wise t and joint Wald statistic.\n    "
    B = d['P'] @ Y
    b = B[d['idx_clase'], :]
    U = Y - d['X'] @ B
    Z = np.add.reduceat(d['w_clase'][:, None] * U, d['inicios'], axis=0)
    V = (Z.T @ Z) * d['correccion']
    se = np.sqrt(np.diag(V))
    t = b / se
    W = float(b @ np.linalg.solve(V, b))
    return b, se, t, W

def wild_bootstrap(d, B=B_BOOTSTRAP, semilla=SEMILLA):
    '\n    Null-imposed wild cluster bootstrap with cluster-level Rademacher weights.\n    '
    rng = np.random.default_rng(semilla)
    b_obs, se_obs, t_obs, W_obs = estadisticos(d['Y'], d)
    J = d['Y'].shape[1]
    T_boot = np.empty((B, J))
    W_boot = np.empty(B)
    tamanos = np.diff(np.r_[d['inicios'], d['n']])
    for i in range(B):
        pesos_cluster = rng.choice([-1.0, 1.0], size=d['G'])
        pesos = np.repeat(pesos_cluster, tamanos)
        Y_estrella = d['ajuste_restringido'] + pesos[:, None] * d['residuos_restringidos']
        _, _, t_b, W_b = estadisticos(Y_estrella, d)
        T_boot[i] = t_b
        W_boot[i] = W_b
    p_por_var = ((np.abs(T_boot) >= np.abs(t_obs)).sum(0) + 1) / (B + 1)
    p_conjunto = float(((W_boot >= W_obs).sum() + 1) / (B + 1))
    return b_obs, se_obs, t_obs, T_boot, p_por_var, p_conjunto

def romano_wolf(t_obs, T_boot, etiquetas):
    '\n    Romano-Wolf stepdown max-|t| using the null-imposed bootstrap.\n    '
    absT = np.abs(T_boot)
    abst = np.abs(t_obs)
    orden = np.argsort(-abst)
    p_ajustados = np.empty(len(etiquetas))
    previo = 0.0
    restantes = list(orden)
    for j in orden:
        max_T = absT[:, restantes].max(axis=1)
        p = float(((max_T >= abst[j]).sum() + 1) / (absT.shape[0] + 1))
        p = max(p, previo)
        p_ajustados[j] = p
        previo = p
        restantes = restantes[1:]
    return {etiquetas[j]: round(p_ajustados[j], 4) for j in range(len(etiquetas))}

def equivalencia_tost(b, se, etiquetas):
    '\n    TOST with reference margins of 10% and 20%, plus a one-sided 95% upper bound.\n    '
    salida = {}
    for j, c in enumerate(etiquetas):
        pruebas = {}
        for nombre, delta_pct in deltas_referencia:
            # Symmetric percentage margins require asymmetric log boundaries.
            limite_bajo = np.log1p(-delta_pct / 100.0)
            limite_alto = np.log1p(delta_pct / 100.0)
            t_bajo = (b[j] - limite_bajo) / se[j]
            t_alto = (b[j] - limite_alto) / se[j]
            p_eq = max(1 - stats.norm.cdf(t_bajo), stats.norm.cdf(t_alto))
            pruebas[nombre] = {'delta_pct': delta_pct, 'p_equivalencia': round(float(p_eq), 4),
                               'rechaza_ese_tamano': bool(p_eq < 0.05)}
        # One-sided 95% upper limit on the SIGNED effect:
        # beta + 1.645 SE, transformed to percent; below the two-sided 95% upper endpoint.
        cota = (np.exp(b[j] + 1.645 * se[j]) - 1) * 100
        salida[c] = {'cota_sup_95_pct': round(float(cota), 1), 'tost': pruebas}
    return salida

print('Functions ready')

Functions ready


## 3. Running both designs under both imputations

In [3]:
resultados = {}
for nombre_dataset in ['hibrido', 'MIMA']:
    panel = pd.read_csv(f'../data_processed/panel_pico_{nombre_dataset}.csv', parse_dates=['date'])
    gaps = pd.read_csv(f'../data_processed/gaps_franjas_{nombre_dataset}.csv', parse_dates=['date'])
    bloques = {'M1_SUR': (panel, [f'l_{c}' for c in contaminantes], 'l_'),
               'M2_franjas': (gaps, [f'gap_{c}' for c in contaminantes], 'gap_')}
    for nombre_bloque, (df, columnas_y, prefijo) in bloques.items():
        d = preparar(df, columnas_y, 'wk')
        b, se, t, T_boot, p_var, p_conjunto = wild_bootstrap(d)
        etiquetas = [col.replace(prefijo, '') for col in columnas_y]
        rw = romano_wolf(t, T_boot, etiquetas)
        eq = equivalencia_tost(b, se, etiquetas)
        clave = f'{nombre_bloque}_{nombre_dataset}'
        resultados[clave] = {
            'wild_bootstrap': {'B': B_BOOTSTRAP, 'G_clusters': int(d['G']),
                               'p_conjunto': round(p_conjunto, 4),
                               'por_var': {etiquetas[j]: {'pct': round((np.exp(b[j]) - 1) * 100, 2),
                                                          'p_boot': round(float(p_var[j]), 4)}
                                           for j in range(len(etiquetas))}},
            'romano_wolf': rw, 'equivalencia': eq}

        print("=" * 70)
        print(f"{nombre_bloque} — {nombre_dataset}")
        print("=" * 70)
        print(f"Joint p (wild bootstrap): {p_conjunto:.3f}")
        for j, c in enumerate(etiquetas):
            e = eq[c]['tost']['ref_10pct']
            print(f"  {c}: {(np.exp(b[j])-1)*100:+.1f}% | p_boot = {p_var[j]:.3f} | p_RW = {rw[c]} | "
                  f"cota 95% = {eq[c]['cota_sup_95_pct']}% | "
                  f"equiv. vs 10%: p = {e['p_equivalencia']}"
                  f"{' -> REJECTED' if e['rechaza_ese_tamano'] else ''}")

with open(resultados_dir / 'inferencia_fina.json', 'w') as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)
print(f"\nSaved to: {resultados_dir / 'inferencia_fina.json'}")

M1_SUR — hibrido
Joint p (wild bootstrap): 0.442
  CO: +7.2% | p_boot = 0.257 | p_RW = 0.668 | cota 95% = 18.1% | equiv. vs 10%: p = 0.3293
  NO2: +9.8% | p_boot = 0.059 | p_RW = 0.253 | cota 95% = 18.8% | equiv. vs 10%: p = 0.4866
  O3: -7.7% | p_boot = 0.288 | p_RW = 0.668 | cota 95% = 3.6% | equiv. vs 10%: p = 0.3607
  PM10: +5.8% | p_boot = 0.428 | p_RW = 0.668 | cota 95% = 18.0% | equiv. vs 10%: p = 0.2785
  PM2.5: +1.3% | p_boot = 0.830 | p_RW = 0.83 | cota 95% = 10.7% | equiv. vs 10%: p = 0.0627


M2_franjas — hibrido
Joint p (wild bootstrap): 0.005
  CO: +4.6% | p_boot = 0.005 | p_RW = 0.021 | cota 95% = 7.2% | equiv. vs 10%: p = 0.0004 -> REJECTED
  NO2: +1.0% | p_boot = 0.834 | p_RW = 0.988 | cota 95% = 7.5% | equiv. vs 10%: p = 0.0117 -> REJECTED
  O3: -2.6% | p_boot = 0.646 | p_RW = 0.979 | cota 95% = 6.7% | equiv. vs 10%: p = 0.078
  PM10: -0.5% | p_boot = 0.853 | p_RW = 0.988 | cota 95% = 4.0% | equiv. vs 10%: p = 0.0001 -> REJECTED
  PM2.5: +0.5% | p_boot = 0.850 | p_RW = 0.988 | cota 95% = 4.9% | equiv. vs 10%: p = 0.0002 -> REJECTED


M1_SUR — MIMA
Joint p (wild bootstrap): 0.477
  CO: +6.8% | p_boot = 0.288 | p_RW = 0.707 | cota 95% = 17.8% | equiv. vs 10%: p = 0.309
  NO2: +9.7% | p_boot = 0.062 | p_RW = 0.266 | cota 95% = 18.7% | equiv. vs 10%: p = 0.4759
  O3: -7.0% | p_boot = 0.329 | p_RW = 0.707 | cota 95% = 4.2% | equiv. vs 10%: p = 0.315
  PM10: +5.8% | p_boot = 0.422 | p_RW = 0.707 | cota 95% = 18.1% | equiv. vs 10%: p = 0.2807
  PM2.5: +1.2% | p_boot = 0.851 | p_RW = 0.851 | cota 95% = 10.8% | equiv. vs 10%: p = 0.0656


M2_franjas — MIMA
Joint p (wild bootstrap): 0.008
  CO: +4.4% | p_boot = 0.009 | p_RW = 0.029 | cota 95% = 7.0% | equiv. vs 10%: p = 0.0003 -> REJECTED
  NO2: +0.8% | p_boot = 0.885 | p_RW = 0.983 | cota 95% = 7.3% | equiv. vs 10%: p = 0.0112 -> REJECTED
  O3: -2.1% | p_boot = 0.708 | p_RW = 0.983 | cota 95% = 7.2% | equiv. vs 10%: p = 0.0636
  PM10: -0.7% | p_boot = 0.809 | p_RW = 0.983 | cota 95% = 3.8% | equiv. vs 10%: p = 0.0001 -> REJECTED
  PM2.5: +1.3% | p_boot = 0.656 | p_RW = 0.983 | cota 95% = 6.0% | equiv. vs 10%: p = 0.0013 -> REJECTED

Saved to: ../data_processed/resultados_metodos/inferencia_fina.json


## 4. Plot: one-sided upper limits for each pollutant

In [4]:
fig, ax = plt.subplots(figsize=(13, 6.5))

eq_franjas = resultados['M2_franjas_hibrido']['equivalencia']
x = np.arange(len(contaminantes))

cotas = [eq_franjas[c]['cota_sup_95_pct'] for c in contaminantes]
bars = ax.bar(x, cotas, 0.55, label='95% upper bound (within-day gaps, weekly-clustered SE)',
              color='#3498db', alpha=0.85, edgecolor='black')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'{bar.get_height():.1f}%', ha='center', va='bottom',
            fontsize=10, fontweight='bold')

ax.axhline(10, color='#e67e22', linestyle='--', linewidth=1.6, label='10% reference margin')
ax.axhline(20, color='#c0392b', linestyle='--', linewidth=1.6, label='20% reference margin')

for j, c in enumerate(contaminantes):
    marcas = []
    if eq_franjas[c]['tost']['ref_10pct']['rechaza_ese_tamano']:
        marcas.append('10%')
    if eq_franjas[c]['tost']['ref_20pct']['rechaza_ese_tamano']:
        marcas.append('20%')
    if marcas:
        ax.text(j, max(cotas[j] + 2.2, 11), 'equivalent within\n' + ' & '.join(marcas), ha='center',
                fontsize=8.5, fontweight='bold', color='#8c2f2f')

ax.set_xticks(x)
ax.set_xticklabels(contaminantes)
ax.set_xlabel('Pollutant', fontsize=12, fontweight='bold')
ax.set_ylabel('One-sided upper bound on the signed effect (%)', fontsize=12, fontweight='bold')
ax.set_ylim(0, 24)
ax.set_title('School-term effect on the morning/control ratio: one-sided 95% upper bounds\n(TOST separately evaluates equivalence within the chosen reference margins)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(graficos_dir / '04_cotas_superiores.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {graficos_dir / '04_cotas_superiores.png'}")

print('\nCOMPLETED')

Plot saved to: ../data_processed/graficos_metodos/04_cotas_superiores.png

COMPLETED
